# C-MAPSS Turbofan Engine Analysis — Fleet-Derived Priors

The NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation) dataset is the standard benchmark for turbofan engine RUL prediction. FD001 contains 100 training engines and 100 test engines operating under a single operating condition with a single fault mode (HPC degradation). This notebook evaluates the adaptive PID framework with a fleet-derived prior — the strongest prior type available.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Colorblind-friendly palette
CB = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12})

FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR = Path("../analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

from datasets.cmapss.config import CMAPSS_CONFIG
from datasets.cmapss.loader import CMAPSSLoader
from datasets.cmapss.feature_extraction import load_cmapss_data, compute_health_index, compute_fleet_prior
from framework.benchmark_runner import run_single_trajectory
from core.adaptive_drift import adaptive_drift_pid, adaptive_drift_with_regime, PIDParams

## 2. Load C-MAPSS FD001

In [ ]:
loader = CMAPSSLoader("FD001")
trajectories = loader.load_trajectories()

train_trajs = [t for t in trajectories if t.metadata["split"] == "train"]
test_trajs  = [t for t in trajectories if t.metadata["split"] == "test"]

print(f"FD001: {len(train_trajs)} training engines, {len(test_trajs)} test engines")
print(f"Informative sensors used: {CMAPSS_CONFIG['informative_sensors']}")
print(f"RUL cap: {CMAPSS_CONFIG['rul_cap']} cycles")
print(f"Prior type: {train_trajs[0].oem_prior.confidence}")
print(f"Fleet prior length: {len(train_trajs[0].oem_prior.baseline_curve)} cycles (= mean training engine life)")

# Quick stats
train_lengths = [len(t.features) for t in train_trajs]
test_lengths  = [len(t.features) for t in test_trajs]
print(f"\nTraining engine duration: min={min(train_lengths)}, median={np.median(train_lengths):.0f}, max={max(train_lengths)} cycles")
print(f"Test engine duration:     min={min(test_lengths)}, median={np.median(test_lengths):.0f}, max={max(test_lengths)} cycles")

## 3. Fleet Prior Visualization

In [ ]:
# Retrieve fleet prior from first trajectory (same for all)
fleet_prior = train_trajs[0].oem_prior.baseline_curve
fleet_cycles = np.arange(len(fleet_prior))

fig, ax = plt.subplots(figsize=(11, 5))

# Plot a sample of individual training trajectories (faded)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(train_trajs), size=min(20, len(train_trajs)), replace=False)
for i in sample_idx:
    traj = train_trajs[i]
    n = len(traj.features)
    hi = traj.features["health_index"].values
    ax.plot(np.arange(n), hi, color=CB[2], alpha=0.18, linewidth=0.7)

# Fleet prior (mean degradation curve)
ax.plot(fleet_cycles, fleet_prior, color=CB[0], linewidth=2.2, label="Fleet prior (mean HI)", zorder=5)

ax.axhline(0.0, color="black", linestyle="--", linewidth=0.9, label="Failure threshold (HI = 0)")
ax.set_xlabel("Cycle")
ax.set_ylabel("Health Index (normalised, 1 = healthy)")
ax.set_xlim(0, len(fleet_prior) + 10)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="upper right")
ax.set_title("")  # no default title

fig.tight_layout()
fig.savefig(FIGURES_DIR / "cmapss_fleet_prior.png", dpi=150, bbox_inches="tight")
plt.show()
print("Fleet prior: average health index degradation across all 100 FD001 training engines.")

## 4. PID Tracking — Example Test Engines

In [ ]:
# Pick 4 test engines: short, medium-short, medium-long, long
test_lengths_arr = np.array([len(t.features) for t in test_trajs])
percentiles = [10, 35, 65, 90]
selected_indices = [
    int(np.argmin(np.abs(test_lengths_arr - np.percentile(test_lengths_arr, p))))
    for p in percentiles
]
# Deduplicate
seen = set()
unique_selected = []
for idx in selected_indices:
    if idx not in seen:
        unique_selected.append(idx)
        seen.add(idx)
selected_test = [test_trajs[i] for i in unique_selected]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes_flat = axes.flatten()

for ax, traj in zip(axes_flat, selected_test):
    n = len(traj.features)
    hi_obs = traj.features["health_index"].values
    cycles = np.arange(n)

    # Fleet prior baseline (aligned to this engine's length)
    baseline = traj.oem_prior.baseline_curve[:n]

    # Run PID adaptive
    try:
        pid_result = adaptive_drift_pid(hi_obs, baseline, threshold=traj.oem_prior.threshold)
        pred_rul = pid_result.predicted_rul
    except Exception:
        pred_rul = None

    ax.plot(cycles, hi_obs, color=CB[0], linewidth=1.2, label="Observed HI", zorder=4)
    ax.plot(cycles, baseline, color=CB[1], linewidth=1.5, linestyle="--", label="Fleet prior (HI)", zorder=3)

    if pred_rul is not None:
        # Convert predicted RUL to an implied HI (for visual comparison)
        # Normalise by fleet prior length so it stays on [0,1] scale
        fleet_len = traj.oem_prior.expected_life
        hi_pred = np.clip(pred_rul / fleet_len, 0, 1)
        ax.plot(cycles, hi_pred, color=CB[2], linewidth=1.2, linestyle=":", label="PID-predicted HI", zorder=5)

    # Ground truth RUL annotation at last step
    gt_rul = traj.metadata.get("ground_truth_rul_at_end", None)
    unit = traj.metadata["unit_nr"]
    ax.set_xlabel("Cycle")
    ax.set_ylabel("Health Index")
    label_str = f"Engine {unit}  (n={n} cycles"
    if gt_rul is not None:
        label_str += f", true RUL={int(gt_rul)})"
    else:
        label_str += ")"
    ax.set_title(label_str, fontsize=10)
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("PID Tracking vs. Fleet Prior — 4 Representative Test Engines (FD001)", fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cmapss_pid_tracking_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Test Set Results — RMSE and NASA S-Score

In [ ]:
# Run benchmark on test trajectories only (ground truth is the true RUL at last step)
test_results_all = []
for traj in test_trajs:
    res = run_single_trajectory(traj)
    test_results_all.append(res)

test_results_df = pd.concat(test_results_all, ignore_index=True)

# Also run on train for completeness
train_results_all = []
for traj in train_trajs:
    res = run_single_trajectory(traj)
    train_results_all.append(res)

train_results_df = pd.concat(train_results_all, ignore_index=True)

# Focus on test-set RMSE / NASA score
test_agg = (
    test_results_df
    .groupby("model")[["rmse", "mae", "nasa_score"]]
    .agg(["mean", "std"])
    .round(2)
)
print("Test set results (FD001, 100 engines):")
print(test_agg.to_string())

# Extract PID results for comparison table
pid_test = test_results_df[test_results_df["model"].isin(["pid_adaptive", "pid_regime"])]
pid_summary = pid_test.groupby("model")[["rmse", "nasa_score"]].mean().round(2)
print("\nPID models — mean test-set RMSE and NASA score:")
print(pid_summary.to_string())

## 6. Published Methods Comparison Table

In [ ]:
# Retrieve PID and PID+Regime numbers dynamically
def _get(df, model, metric):
    row = df[df["model"] == model]
    if row.empty or row[metric].isna().all():
        return float("nan")
    return round(float(row[metric].mean()), 2)

pid_rmse    = _get(pid_summary.reset_index(), "pid_adaptive", "rmse")
pid_score   = _get(pid_summary.reset_index(), "pid_adaptive", "nasa_score")
regime_rmse = _get(pid_summary.reset_index(), "pid_regime",   "rmse")
regime_score= _get(pid_summary.reset_index(), "pid_regime",   "nasa_score")

comparison = pd.DataFrame([
    {"Method": "PID Adaptive",  "Type": "Physics-informed (zero-training)", "FD001 RMSE": pid_rmse,    "FD001 Score": pid_score,    "Source": "This work"},
    {"Method": "PID + Regime",  "Type": "Physics-informed (zero-training)", "FD001 RMSE": regime_rmse, "FD001 Score": regime_score, "Source": "This work"},
    {"Method": "LSTM",          "Type": "Deep learning",                    "FD001 RMSE": 16.14,       "FD001 Score": 338,          "Source": "Zheng et al. 2017"},
    {"Method": "CNN-LSTM",      "Type": "Deep learning",                    "FD001 RMSE": 12.83,       "FD001 Score": 222,          "Source": "Li et al. 2020"},
    {"Method": "Transformer",   "Type": "Deep learning",                    "FD001 RMSE": 11.87,       "FD001 Score": 197,          "Source": "Mo et al. 2021"},
])

print("FD001 Benchmark Comparison:")
print(comparison.to_string(index=False))

In [ ]:
# Visual comparison: RMSE bar chart
fig, ax = plt.subplots(figsize=(10, 5))

methods = comparison["Method"].tolist()
rmse_vals = comparison["FD001 RMSE"].tolist()
colors = [CB[0], CB[1], CB[4], CB[4], CB[4]]  # blue for PID, lighter for DL
edge_colors = ["black" if "This work" in s else "none" for s in comparison["Source"]]

bars = ax.barh(methods[::-1], rmse_vals[::-1], color=colors[::-1],
               edgecolor=edge_colors[::-1], linewidth=1.2)
for bar, val in zip(bars, rmse_vals[::-1]):
    ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=10)

ax.set_xlabel("RMSE (cycles)")
ax.set_ylabel("")
ax.set_xlim(0, max(rmse_vals) * 1.25)
ax.axvline(0, color="black", linewidth=0.5)

# Annotation: zero-training vs. trained
ax.text(0.98, 0.04, "Bold border = this work (zero training data)",
        transform=ax.transAxes, ha="right", fontsize=9, color="gray")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "cmapss_rmse_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Discussion: Where Does PID Sit?

The adaptive PID approach is **zero-training**: it requires no labelled engine failures, no gradient descent, and no hyperparameter grid search. At test time it runs in O(1) per cycle with fixed arithmetic operations. This puts it in a fundamentally different category from LSTM, CNN-LSTM, and Transformer baselines.

**Computational profile:**
| Property | PID Adaptive | Deep Learning |
|----------|-------------|---------------|
| Training data required | None (fleet prior only) | Full training set |
| Training time | None | Minutes–hours |
| Inference complexity | O(1) per cycle | O(n × d) per step |
| Memory at inference | ~5 floats (PID state) | Millions of parameters |
| Explainability | Full (proportional / integral / derivative terms) | Partial (attention maps) |

**Expected RMSE positioning:**
- Deep learning methods benefit from fitting the exact shape of the C-MAPSS degradation curve from 100 labelled run-to-failure examples.
- The PID tracker uses only the fleet average curve. It will underperform deep learning on this in-distribution benchmark.
- The real advantage emerges in **out-of-distribution settings** (new engine variants, early deployment with <5 historical failures) where deep learning cannot be reliably trained.

## 8. Save Results

In [ ]:
all_cmapss = pd.concat([train_results_df, test_results_df], ignore_index=True)
output_path = ANALYSIS_DIR / "cmapss_metrics.csv"
all_cmapss.to_csv(output_path, index=False)
print(f"Saved {len(all_cmapss)} result rows to {output_path}")

# Save comparison table
comparison_path = ANALYSIS_DIR / "cmapss_published_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print(f"Saved published comparison to {comparison_path}")

print("\nFinal summary:")
print(f"  PID Adaptive — RMSE: {pid_rmse}, NASA Score: {pid_score}")
print(f"  PID + Regime — RMSE: {regime_rmse}, NASA Score: {regime_score}")
print(f"  Best deep learning (Transformer) — RMSE: 11.87, Score: 197")
print(f"\n  Zero-training gap: {pid_rmse - 11.87:+.2f} RMSE cycles vs. Transformer")